# Lilly v2 — listener pass-2 half 2 (resume epoch 2)

Attach **half 1's Output** (`lilly-listen-half1.zip`) as a kernel source before
Save & Run All. This notebook clones to `/kaggle/temp`, re-downloads the mix,
and runs `train_speech.py --resume` with `SPEECH_EPOCHS = 2` so Hugging Face
Trainer continues the same LoRA (optimizer + schedule intact).

`--base` on a merged `listen-trained` folder would start a **new** adapter.
That is not this notebook.

No BEFORE WER (same untrained large-v3 every time). AFTER WER runs after
convert and **before** `lilly-listen.zip`. If scoring fails, there is no
app zip.

Set these in the panel on the right:

- **Session options → Accelerator → GPU T4**
- **Session options → Internet → On**
- **Add data → Kernel output → lilly-speech** (must contain `lilly-listen-half1.zip`)

Then **Save Version → Save & Run All (Commit)** and close the tab.


In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co", "https://datasets-server.huggingface.co"):
    reachable(host)
print("network ok")

def run(*cmd):
    """Run a step and let a failure actually stop the notebook."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)


In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working
# Everything under /kaggle/working becomes Output. A clone there floods
# `kaggle kernels output` with git objects so the zip never downloads (OCR
# already learned this). Only the zip(s) below belong in Output.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "train_speech.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zips")


In [ ]:
# 3. Install what we need (~3 min)
# The versions are read out of the repo's own requirements.txt rather than
# copied into this cell. A second, hand-kept list is exactly how the last run
# died: peft is pinned in requirements.txt and was simply absent from here, so
# Kaggle's own much newer peft got used instead — and that one's torchao
# dispatcher raises against the torchao Kaggle also ships. The first
# get_peft_model() call blew up, after a 3 GB download and forty minutes.
NEEDED = ["transformers", "accelerate", "peft", "faster-whisper", "ctranslate2",
          "soundfile", "scipy", "pyarrow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
unpinned = [n for n in NEEDED if n not in pins]
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin in requirements.txt, taking latest:", unpinned or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])


In [ ]:
# 3b. Prove this machine can actually train, before an hour is spent finding out
# Two things have to hold and neither shows up in a version number: peft must be
# able to build a LoRA layer on this image, and the GPU Kaggle handed us must
# actually run kernels. The last run satisfied "GPU is available" and still could
# not compute — Kaggle gave a P100 (sm_60) that the installed PyTorch does not
# support, and separately peft could not build a layer at all.
#
# So: build a real LoRA layer, put it on the GPU, push a gradient through it. It
# takes about twenty seconds and it fails here, loudly, instead of after the
# download.
import torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__)

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(32, 32)
    def forward(self, x):
        return self.q_proj(x)

tiny = get_peft_model(Tiny(), LoraConfig(r=4, target_modules=["q_proj"]))
try:
    tiny = tiny.cuda()
    out = tiny(torch.randn(4, 32, device="cuda")).sum()
    out.backward()
except RuntimeError as exc:
    raise SystemExit(
        f"The GPU cannot run this build of PyTorch ({exc}).\n"
        f"Card: {torch.cuda.get_device_name(0)}. Right panel -> Session options ->"
        f" Accelerator -> GPU T4 x2, then Save & Run All again.") from exc

lora = [n for n, p in tiny.named_parameters() if "lora_" in n and p.grad is not None]
assert lora, "peft built a LoRA layer but no gradient reached it"
print(f"LoRA trains on {torch.cuda.get_device_name(0)}: "
      f"{len(lora)} adapter tensors took a gradient")
del tiny
torch.cuda.empty_cache()


In [ ]:
# 4. Download the speech: clips to train on, and clips held back to judge with (~3 GB)
# The audio goes to scratch space, not into /kaggle/working — everything in the working
# directory is copied into the version Output, and 3 GB of wav files there is waste.
scratch = Path("/kaggle/temp/speech" if Path("/kaggle/temp").is_dir() else "/tmp/lilly-speech")
scratch.mkdir(parents=True, exist_ok=True)
if Path("data/speech").is_symlink():
    Path("data/speech").unlink()          # re-running the cell should not fail
if not Path("data/speech").exists():
    Path("data/speech").symlink_to(scratch)

run("python3", "data/scripts/download_speech_data.py")

for split in ("train", "valid", "test"):
    n = sum(1 for _ in open(f"data/speech/{split}.tsv", encoding="utf-8"))
    assert n > 100, f"{split}.tsv has only {n} clips — a download failed"
    print(f"{split}: {n:,} clips")


In [ ]:
# 4b. Croatian speech — downloaded here rather than uploaded from home.
# Kaggle's connection runs at roughly twenty times the one this project is
# developed on, and the audio is several gigabytes: pulling it here costs
# minutes where uploading it as a dataset would cost most of an evening.
# Scratch space for the same reason the Bosnian clips use it — anything left in
# /kaggle/working is copied into the Output, and gigabytes of wav there is waste.
extra = Path("/kaggle/temp/speech-extra" if Path("/kaggle/temp").is_dir()
             else "/tmp/lilly-speech-extra")
extra.mkdir(parents=True, exist_ok=True)
if Path("data/speech-extra").is_symlink():
    Path("data/speech-extra").unlink()
if not Path("data/speech-extra").exists():
    Path("data/speech-extra").symlink_to(extra)

# Two sources, two calls: a single --hours would override both defaults.
# fleurs_hr default is already 12 h (the whole train split). voxpopuli_hr
# default is 8 h of spontaneous EP speech FLEURS does not have.
run("python3", "data/scripts/download_extra_speech.py",
    "--source", "fleurs_hr", "--hours", "12")
run("python3", "data/scripts/download_extra_speech.py",
    "--source", "voxpopuli_hr")

fleurs_n = sum(1 for _ in open("data/speech-extra/fleurs_hr/train.tsv", encoding="utf-8"))
vox_path = Path("data/speech-extra/voxpopuli_hr/train.tsv")
assert fleurs_n > 500, f"only {fleurs_n} FLEURS hr clips — the download did not work"
assert vox_path.is_file(), (
    "voxpopuli_hr missing — this pass is the wider mix, not another FLEURS-only run")
vox_n = sum(1 for _ in open(vox_path, encoding="utf-8"))
assert vox_n > 200, f"only {vox_n} voxpopuli clips — the second source did not land"
print(f"Croatian FLEURS hr: {fleurs_n:,}  voxpopuli_hr: {vox_n:,}")


In [ ]:
# 4c. Same mix as half 1 — Bosnian share 0.47, same two Croatian sources.
BOSNIAN_SHARE = 0.47
# Trainer num_train_epochs=2 + resume from the epoch-1 checkpoint runs epoch 2
# only. SPEECH_EPOCHS = 1 here would look already finished and train nothing.
SPEECH_EPOCHS = 2
SPEECH_BASE = "openai/whisper-large-v3"

run("python3", "data/scripts/build_speech_mix.py", "--share", str(BOSNIAN_SHARE))

MIX = "data/speech-extra/train-mix.tsv"
bosnian_only = sum(1 for _ in open("data/speech/train.tsv", encoding="utf-8"))
mixed = sum(1 for _ in open(MIX, encoding="utf-8"))
assert mixed > bosnian_only, (
    f"the mix has {mixed:,} rows against {bosnian_only:,} Bosnian — no Croatian "
    f"got in, so this run would not test what it is here to test")
print(f"mixed: {mixed:,} rows, from {bosnian_only:,} Bosnian, "
      f"resume to {SPEECH_EPOCHS} epochs")


In [ ]:
# 4d. Half 1's Trainer checkpoint, from the attached kernel Output (or a dataset).
# Zip contents are models/lilly/listen-adapter/… relative to the unzip root.
hits = list(Path("/kaggle/input").rglob("lilly-listen-half1.zip")) if Path("/kaggle/input").is_dir() else []
assert hits, (
    "lilly-listen-half1.zip is not attached. Add the Output of the lilly-speech "
    "kernel (half 1) as a kernel source, or upload the zip as a dataset.")
extract = SCRATCH / "half1"
extract.mkdir(parents=True, exist_ok=True)
run("unzip", "-o", "-q", str(hits[0]), "-d", str(extract))
adapter = extract / "models" / "lilly" / "listen-adapter"
if not (adapter / "trainer_state.json").is_file():
    found = list(extract.rglob("trainer_state.json"))
    assert found, f"zip is not a Trainer checkpoint: {hits[0]}"
    adapter = found[0].parent
print("resuming", adapter)


In [ ]:
# 5. Epoch 2 of the same run, convert, score, then zip the app listener.
# --resume the half-1 checkpoint. --epochs 2 so Trainer is not already done.
# Adapter zip is the resume checkpoint, not the product. AFTER WER must run
# before lilly-listen.zip: a listener we did not measure is not shippable.
run("python3", "training/train_speech.py", "--data", MIX, "--base", SPEECH_BASE,
    "--epochs", str(SPEECH_EPOCHS), "--batch-size", "1", "--grad-accum", "16",
    "--resume", str(adapter), "--no-convert")
assert (Path("models/lilly/listen-adapter") / "trainer_state.json").is_file(), (
    "epoch 2 wrote no Trainer checkpoint")
run("zip", "-qr", "/kaggle/working/lilly-listen-half2.zip",
    "models/lilly/listen-adapter")
print("saved adapter zip before convert")

trained = Path("models/lilly/listen-trained")
weight_files = (
    list(trained.glob("model*.safetensors"))
    + list(trained.glob("pytorch_model*.bin"))
    + list(trained.glob("*.safetensors"))
)
assert weight_files, (
    f"nothing was merged under {trained}: "
    f"{sorted(p.name for p in trained.iterdir()) if trained.is_dir() else 'missing'}")
print("merged weights:", ", ".join(p.name for p in weight_files))

run("python3", "training/train_speech.py",
    "--base", "models/lilly/listen-trained", "--convert-only", "models/lilly/listen")
assert Path("models/lilly/listen/model.bin").is_file(), "conversion produced nothing"
run("python3", "training/evaluate_speech.py", "--data", "data/speech/test.tsv",
    "--model", "models/lilly/listen", "--limit", "200", "--show", "3")
run("zip", "-qr", "/kaggle/working/lilly-listen.zip", "models/lilly/listen")
size = Path("/kaggle/working/lilly-listen.zip").stat().st_size
assert size > 1_000_000, f"the zip is only {size} bytes"
print(f"lilly-listen.zip — {size / 1048576:.0f} MB")


**Done.** Download `lilly-listen.zip` from the **Output** tab and unzip it over
`models/lilly/listen/`. Keep a copy of the listener you are replacing first.

`lilly-listen-half2.zip` is the epoch-2 Trainer checkpoint if you need to
resume further. AFTER WER ran in this notebook before that zip was written.
If scoring failed, there is no `lilly-listen.zip`.
